In [ ]:
#importing important libraries
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score
import pandas as pd

In [ ]:
#reading csv file and preprocessing
df =pd.read_csv('C:/Users/Lenovo/Downloads/college_sleep_and_gpa.csv')
df['first_generation']=df['first_generation'].fillna(df['first_generation'].mode()[0])
df['term_units']=df['term_units'].fillna(df['term_units'].mean())
df['term_load_z']=df['term_load_z'].fillna(df['term_load_z'].mean())
df=df.drop(['sleep_midpoint_clock'],axis=1)
df=df.dropna()


In [ ]:
#train/test split
X =df.drop(['term_gpa', 'gpa_change', 'student_id'], axis=1)
y =df['term_gpa']

X_train, X_test, y_train, y_test =train_test_split(X, y, test_size=0.2, random_state=20)


In [30]:
#scaling and encoding
encoder = OneHotEncoder(handle_unknown='ignore')
scalar_X = StandardScaler()

preprocessor = ColumnTransformer([
    ('num', scalar_X, [ 'study','first_generation', 'underrepresented', 'avg_sleep_minutes',
       'avg_sleep_hours', 'daytime_sleep_minutes', 'sleep_midpoint_minutes',
       'bedtime_variability', 'nights_tracked_fraction', 'prior_gpa',
         'term_units', 'term_load_z', 'under_6h_sleep',]),
    
    ('cat', encoder, ['university', 'semester', 'cohort_code', 'gender', 'sleep_bracket'])
    
])

X_train_prc = preprocessor.fit_transform(X_train)
X_test_prc = preprocessor.transform(X_test)

scalar_Y = StandardScaler()

y_train_scaled= scalar_Y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled=scalar_Y.transform(y_test.values.reshape(-1, 1))


In [32]:
# Random Forest

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_prc, y_train_scaled)
scores =cross_val_score(model, X_train_prc, y_train_scaled, cv=5, scoring='r2')
print(scores)
print('Mean',scores.mean())
print("std", scores.std())


c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:1403: DataCon

[0.3205599  0.41641038 0.49945983 0.14694391 0.35781327]
Mean 0.3482374594989716
std 0.11738193291823819


The results from the cross validation of random forest model
[0.32142234 0.43853181 0.41068996 0.00472164 0.30233108]
Mean 0.2955393688855267
std 0.15427086823971256

 Mean is low with high std in comparision to the mean, and we have clear outlier, that means the perforemence is not consistent and weak 
 
 comparing with day 1
 R2 0.48062073520535487
 RSME 0.7596894652457978
 the R2 here is much higher than the mean we got with 5-folds cross validation, since the single split only reflects one random division of the data, it probably overestimated how well the model generlizes.

 The cross validation is more trusted since it tries many splits over the data sets and showa that the real model is weak and not reliable

In [ ]:
#step 4
scalar2 =StandardScaler()
X, y= load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, random_state=42)

X_train_scaled =scalar2.fit_transform(X_train)
X_test_scaled =scalar2.fit_transform(X_test)

model=LogisticRegression(max_iter=1000)
model.fit(X_train_scaled,y_train)

In [35]:
#stratified Kfold
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='f1')
print(scores)

[0.96610169 0.95652174 0.98214286 0.99130435 0.95726496]


stratified Kfolds we can use them in classification so we avoid random slicing and make sure each fold has the same amount of classes so we can have high mean between scores and low varience and trust the model for future inseen data